# Training diagnostics across seeds (2-asset base model)

Overlays the base model (**Seed 1** = `PPO_2Assets_Trial1`) and the four re-runs
(**Seed 2-5** = `PPO_2Assets_Trial1_seed{2..5}`) for three PPO training metrics,
exported from TensorBoard as CSV (`Wall time, Step, Value`):

* **Mean Episodic Reward** -> `N_figures/2Assets/mean_episodic_reward/`
* **KL Divergence**        -> `N_figures/2Assets/approx_kl/`
* **Explained Variance**   -> `N_figures/2Assets/train_explained_var_Graph/`

Publication-quality styling matched to `N_contor_plot_*` and `*_PnL_Simulations`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ---------- project root ----------
try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    HERE = os.getcwd()
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if PROJECT_ROOT is None:
    p = HERE
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "N_figures")):
            PROJECT_ROOT = p; break
        p = os.path.dirname(p)
    if PROJECT_ROOT is None:
        PROJECT_ROOT = os.path.dirname(HERE)
FIG_ROOT = os.path.join(PROJECT_ROOT, "N_figures", "2Assets")
print("PROJECT_ROOT:", PROJECT_ROOT)

# ---------- options ----------
SAVE   = True          # save combined.pdf/png into each metric's folder
SMOOTH = 0.0           # TensorBoard-style EMA weight in [0,1); 0.0 = raw curves (e.g. 0.6 to smooth)

SEEDS  = [1, 2, 3, 4, 5]
SEED_COLORS = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c", 4: "#d62728", 5: "#9467bd"}

# ---------- academic style (matched to N_contor_plot_* and *_PnL_Simulations) ----------
PAPER_RC = {
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "axes.titlesize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
}

XLABEL = "Environment Interaction Steps"

# ---------- data-path resolver ----------
def csv_path(data_root, seed):
    if seed == 1:
        return os.path.join(data_root, "PPO_2Assets_Trial1", "PPO_2Assets_Trial1_PPO_1.csv")
    return os.path.join(data_root, f"seed{seed}",
                        f"PPO_2Assets_PPO_2Assets_Trial1_seed{seed}_PPO_1.csv")

def _ema(y, weight):
    if not weight:
        return y
    out = np.empty_like(y, dtype=float); last = float(y[0])
    for i, v in enumerate(y):
        last = last * weight + (1.0 - weight) * float(v)
        out[i] = last
    return out

def _millions(x, _pos):
    return "0" if x == 0 else f"{x/1e6:g}M"

def plot_metric(data_root, ylabel, save_dir, fname="combined"):
    plt.rcParams.update(PAPER_RC)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    n_ok = 0
    for seed in SEEDS:
        path = csv_path(data_root, seed)
        if not os.path.exists(path):
            print(f"  [seed {seed}] MISSING: {path}"); continue
        df = pd.read_csv(path).sort_values("Step")
        x = df["Step"].to_numpy(dtype=float)
        y = _ema(df["Value"].to_numpy(dtype=float), SMOOTH)
        ax.plot(x, y, color=SEED_COLORS[seed], linewidth=2.0, alpha=0.9, label=f"Seed {seed}")
        n_ok += 1
    ax.set_xlabel(XLABEL)
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_formatter(FuncFormatter(_millions))
    ax.legend(fontsize=16, ncol=1)
    ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
    ax.tick_params(width=1.5, length=6)
    fig.tight_layout()
    if SAVE and n_ok:
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(os.path.join(save_dir, f"{fname}.pdf"), bbox_inches="tight")
        fig.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=600, bbox_inches="tight")
        print(f"  saved -> {os.path.join(save_dir, fname)}.{{pdf,png}}")
    plt.show()
    print(f"  plotted {n_ok}/{len(SEEDS)} seeds" + ("" if SAVE else "  (SAVE=False, nothing written)"))

In [ ]:
# ---- Mean Episodic Reward ----
plot_metric(
    data_root = os.path.join(FIG_ROOT, "mean_episodic_reward_data"),
    ylabel    = "Mean Episodic Reward",
    save_dir  = os.path.join(FIG_ROOT, "mean_episodic_reward"),
)

In [ ]:
# ---- KL Divergence ----
plot_metric(
    data_root = os.path.join(FIG_ROOT, "approx_kl_data"),
    ylabel    = "KL Divergence",
    save_dir  = os.path.join(FIG_ROOT, "approx_kl"),
)

In [ ]:
# ---- Explained Variance ----
plot_metric(
    data_root = os.path.join(FIG_ROOT, "train_explained_var_Data"),
    ylabel    = "Explained Variance",
    save_dir  = os.path.join(FIG_ROOT, "train_explained_var_Graph"),
)